ReACT agent architecture is mostly used when 2 or more queries are made to the llm in a single prompt

In [1]:
from langchain.messages import HumanMessage,AIMessage,SystemMessage
from langchain.agents import create_agent
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START,END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import add_messages
from tavily import TavilyClient
from langchain.tools import tool
from typing import TypedDict, Annotated
from langgraph.prebuilt import ToolNode, tools_condition

In [2]:
load_dotenv()

True

In [3]:
llm = init_chat_model(model="google_genai:gemini-3.6-flash")

In [4]:
tavily_client = TavilyClient()

@tool
def tavily_search(query: str) -> dict:
    """Search the web for up-to-date information, news, facts, and answers."""
    return tavily_client.search(query=query)


In [5]:
#custom fucntion
@tool
def multiply(a:int, b:int) -> int:
    """multiply a and b
    
    args:
    a: first int
    b: second int
    
    returns:
    int: output int"""
    return a*b

In [6]:
tools = [tavily_search, multiply]
llm_with_tools = llm.bind_tools(tools)
 

In [7]:
llm_with_tools 

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.6-flash', temperature=1.0, client=<google.genai.client.Client object at 0x0000016844DABDD0>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'tavily_search', 'description': 'Search the web for up-to-date information, news, facts, and answers.', 'parameters': {'properties': {'query': {'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'multiply', 'description': 'multiply a and b\n\n    args:\n    a: first int\n    b: second int\n\n    returns:\n    int: output int', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [8]:
class State(TypedDict):
    messages : Annotated[list, add_messages]

In [9]:
#node functionality
def tool_calling_llm(state:State):
    #we are giving this system prompt so that the model does not loop back in aimessage and tool call which it sometimes does which burns token limit
    system = SystemMessage(content="You are a helpful assistant. Use tools when needed but avoid redundant searches. One search should be sufficient for most questions.")
    return {"messages":[llm_with_tools.invoke([system] + state['messages'])]}

In [10]:
#graph
builder =StateGraph(State)

In [11]:
#nodes
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools",ToolNode(tools))

Implementing react architecture here

In [12]:
#edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    #if the latest message from assistant is a tool call -> tool_condition results to tools
    #If the latest message from assitant is not a tool call -> tool_conditions routes to END"""
    tools_condition
)
builder.add_edge("tools","tool_calling_llm")

In [13]:
graph = builder.compile()

In [14]:
graph.invoke({"messages":"what is the recent news about mayor mamdani?"})

{'messages': [HumanMessage(content='what is the recent news about mayor mamdani?', additional_kwargs={}, response_metadata={}, id='a7a2916d-6e30-4dc1-9beb-3253ecb5c4dc'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "mayor mamdani news"}'}, '__gemini_function_call_thought_signatures__': {'call_3538667': 'EusDCugDAWkUfRP5+DmktV9aIF11oZ2M01cVlMXY0Cwzd+oL3N347CkOpXrQfDsvlAV7JSCqIVtIipLLP4b51cr5OfTG7om1hgU7QoyYkPFGhoLH+X+Z4Rl4B1fo9u5ZR8y+iYTfYnmIKS/uU+0vhFAOWzjUQJaUz0UnPAwhcR34rb1AvfvpC2pN6fEUK35Om2VgPWQRir4HYG7TSFtTnbZv7ACv0a62pJ13pTC/JSMq9tVgJxgA70J7S9rwTB+Oxi7krzk5jnwqx0Sua9N5tZ6f2AXX3i7ZC6mV3Od4K8bZiqUVRDXSE4KRQLxI8g7Ivq2GjpZp3HyXI7OO4gfn9A0GDt3hDUjEpKEjY+qcZbpZO+jmOPmcSBbrj6uIMvDHjcvXO7FwAlcxEKbrbueEjK29Zw37m8hrWdCOzrj5DKhkyH/pcgRleIbqny+Qq6AiTmkULw/IvoJuz5EfrUsIafatZvKDbXeJT3S+tT4HJN+8oqRQz85C+ColkTfe8vTj9rIgbFS0algTOsvCYPVL7BYimyn68hIfXDjxfm2J/mOCAogbnv8QJLABT8GY1P0bPPd9O8CMvJ6B0prx5X1NcyYXmMgDUWhwgdMM5EdsuH4xb4/blr4fe

In [15]:
graph.invoke({"messages":"WHO IS THE CURRENT MAYOR OF NEW YORK"})

{'messages': [HumanMessage(content='WHO IS THE CURRENT MAYOR OF NEW YORK', additional_kwargs={}, response_metadata={}, id='a7fde267-a0b1-4b0a-9f49-858236414ea9'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "current mayor of New York City"}'}, '__gemini_function_call_thought_signatures__': {'call_2811734': 'EskDCsYDAWkUfRPgLX4oEtT4tb6BSp/PmcxmrpwldTkzSSJ831LDyVVpv+2LjwpjyylDWy46ag/sTsVHGEiapX5XWvlfwl4WDievDGlhHm8Jmzr2srTbBH6JpHPkWnVa332fM3f5E5VYisnf/fueRgumkPdOBO5ma/Iv36XDAw7d+bhZmEJwfI4Q9rpSMTUZrAqgne/0Is2TfqsMWRjCDeIAZt+fFEmbfjVj0vahUw8LeAZa0d/H/d086j39x7IXPsiOAEjytO+Q+vK4YY6/Z0FcFwwjdg+pbGRQ5qhFApRw4KN9mzltdNbYLx70as0kLZ9zjRYNWWq21ywqh1kMWFiefpT5wGXxl0zxNt8hJSZlRfrPeFzeLBow4Fl3/d+IaXlN1+OplH668swBMX9+lyMvsP8JO82UN2kZ4CK0i4D5vHX/Wmiq79KQUIS8Im3FylTbdb7eeSXtQ2RsmouPbs81Y9siz5C8IGZ8prQEBEwxTmNTOcQzU8xloAFOEuzzczPTZEonUZ/SxviC9zt259l75HyXf79pyZa0IwiXdAJyFBbZu42ffih9OTxv/zJstl5U1rkaN3HmUCJ/LA4eye9QCyQeFmQg3BeoVA=='}}, resp

In [16]:
graph.invoke({"messages":"what is the recent news about anthropic and what is 120 multiplied by 4532"})

{'messages': [HumanMessage(content='what is the recent news about anthropic and what is 120 multiplied by 4532', additional_kwargs={}, response_metadata={}, id='32a66d6f-97a7-401f-a6df-781fb34c34b9'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "Anthropic latest news recent updates 2025"}'}, '__gemini_function_call_thought_signatures__': {'call_3929598': 'EvcDCvQDAWkUfROlV8vpbAR1+xy1WdzgK8Znqkw1o22eUpW6wTMYm82qq7N/mr3grUetSvMLQcgAmIYFT72wYmTtE5HhhNvygYDgW5QKOJ+uG2Yi+fTRbfDm1vVU30E2uC3Z3XgQma2Z5Uo9QNHcxQPkkP4b3b9emVhhGUJ2FlwyTxT0guZ5GxbGdJ9KKxAUZy36uTumoY0jTPy0Nxahs3D4Sm+iM7CQTuHjH6iTq5bFRdluWVi4axJ0YfYcriK7KRwbGsjqu6wfbxGoU/c72ZwnFy+SKJfc6pABu5Hq1yt5ZuBZofDETo+joCeEpvmttYBncIHmANBYpl4QvZagBMrUV8cqQqwNn9cKzVfaliSRd1uPa0Kt1cTiPyT2gsOmQzefuTgv9nHTgLeWefwHO6KwnhK4iMNco/YTfjqD0Qt1CrzjqsIKZZu7NZ3xeXapv986mwyjvsysJrlX/LeCVNiJox3ltQblOZ0PO0L1YR977msrmS7KUpzNwlKBDULG6f5cBqoIf9aTm9wygUQSZTUyDcaLnbdOqQif1AmBN7T1QjY230junoqzMomguqJs

In [18]:
response = graph.invoke({"messages":"what is the recent news about anthropic and what is 120 multiplied by 4532"})
for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

what is the recent news about anthropic and what is 120 multiplied by 4532
================================== Ai Message ==================================

[]
Tool Calls:
  multiply (call_3763922)
 Call ID: call_3763922
  Args:
    b: 4532
    a: 120
  tavily_search (call_3763923)
 Call ID: call_3763923
  Args:
    query: Anthropic recent news
================================= Tool Message =================================
Name: multiply

543840
================================= Tool Message =================================
Name: tavily_search

{"query": "Anthropic recent news", "follow_up_questions": null, "answer": null, "images": [], "results": [{"title": "EXCLUSIVE: Anthropic quietly sets up biology lab as it ramps AI drug program", "url": "https://www.reuters.com/world/anthropic-quietly-sets-up-biology-lab-it-ramps-ai-drug-program-2026-09-18", "content": "Anthropic has looked to hire a leader who c